# SPORTLIS U-Net Extended — Analysis & Basin Time Series

Analisi dei risultati del modello U-Net esteso (1985–2024, dominio 484×698 @ 3km):
1. XAI permutation importance barplot
2. Performance per bacino (Cascades, Sierra Nevada, Rockies)
3. Time series SWE predetto vs SPORTLIS per bacino
4. Proiezione 2025–2026

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import torch
import torch.nn.functional as F
import netCDF4 as nc
import warnings, gc
from pathlib import Path
from torch import nn
from torch.utils.data import Dataset, DataLoader
from scipy.ndimage import uniform_filter

warnings.filterwarnings('ignore')

# ── Auto-detect environment ───────────────────────────────────────────────────
_ON_NCAR  = Path('/glade/u/home/cionni').exists()
_ON_SMCE  = (not _ON_NCAR) and Path('/home/jovyan/efs').exists()

if _ON_NCAR:
    NCAR_HOME         = Path('/glade/u/home/cionni')
    NCAR_SCRATCH      = NCAR_HOME / 'derecho_scratch'
    PROJECT_DIR       = NCAR_HOME / 'unet_sportlis'
    SPORTLIS_DATA_DIR = NCAR_SCRATCH / 'sportlis_swe'
    ZARR_DIR          = NCAR_SCRATCH / 'zarr_extended'
    MEMMAP_DIR        = NCAR_SCRATCH / 'memmap_extended'
    OUTPUT_DIR        = PROJECT_DIR  / 'output_extended'
    AUX_DIR           = PROJECT_DIR  / 'auxiliary'
else:
    PROJECT_ROOT      = Path('/Users/irene/PROJECTS.index/Hydrology/projects/sportlis_project')
    SPORTLIS_DATA_DIR = PROJECT_ROOT / 'data/inputs'
    ZARR_DIR          = PROJECT_ROOT / 'prepared_pp_narr_sportlis_extended'
    MEMMAP_DIR        = PROJECT_ROOT / 'memmap_extended'
    OUTPUT_DIR        = Path('sportlis_unet_extended_loyo_outputs')
    AUX_DIR           = PROJECT_ROOT / 'auxiliary'

FIG_DIR = OUTPUT_DIR / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

# ── Dominio ───────────────────────────────────────────────────────────────────
EXT_LAT_MIN, EXT_LAT_MAX = 35.005, 49.495
EXT_LON_MIN, EXT_LON_MAX = -124.93, -104.01
EXT_H, EXT_W = 484, 698
lat_1d = np.linspace(EXT_LAT_MIN, EXT_LAT_MAX, EXT_H)
lon_1d = np.linspace(EXT_LON_MIN, EXT_LON_MAX, EXT_W)

# ── Canali ────────────────────────────────────────────────────────────────────
INPUT_VARS         = ['precip_7d','precip_30d','precip_60d','precip_wytd',
                      'tair_30d_mean','tair_30d_max','degree_day_30d']
TOPO_VARS          = ['elevation','slope','aspect_sin','aspect_cos',
                      'tpi_short','tpi_long','canopy_fraction']
CHANNEL_NAMES      = INPUT_VARS + ['lat_norm','lon_norm'] + TOPO_VARS
CHANNEL_GROUP      = ['temporal']*7 + ['coord','coord'] + ['topo']*7
TARGET_VAR         = 'swe_target_filled'
MASK_VAR           = 'swe_mask'
N_IN_CHANNELS      = 16
LOYO_STEP          = 5
YEARS_AVAILABLE    = list(range(1985, 2025))
LOYO_FOLDS         = [{'test': y, 'val': y+1 if y+1 in YEARS_AVAILABLE else y-1}
                       for y in YEARS_AVAILABLE[::LOYO_STEP]]
CHECKPOINT_TEMPLATE = 'best_unet_extended_LOYO_test{test}.pt'
DROPOUT_P           = 0.1

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'OUTPUT_DIR: {OUTPUT_DIR}')
print(f'Folds: {len(LOYO_FOLDS)}')

In [ ]:
# ── Definizione bacini (coordinate geografiche da letteratura) ─────────────────
# Riferimenti: Hamlet & Lettenmaier 2007, Mote et al. 2018, Margulis et al. 2016
# Dominio: lat [35.005, 49.495], lon [-124.93, -104.01]

BASINS = {
    'Pacific_NW_Cascades': {
        'lat': (46.0, 49.5), 'lon': (-124.5, -120.0),
        'color': '#1f77b4', 'label': 'Pacific NW Cascades'
    },
    'Oregon_Cascades': {
        'lat': (42.0, 46.0), 'lon': (-123.0, -120.0),
        'color': '#17becf', 'label': 'Oregon Cascades'
    },
    'Sierra_Nevada': {
        'lat': (36.0, 40.5), 'lon': (-121.5, -117.5),
        'color': '#ff7f0e', 'label': 'Sierra Nevada'
    },
    'Northern_Rockies': {
        'lat': (44.0, 49.5), 'lon': (-117.0, -110.0),
        'color': '#2ca02c', 'label': 'Northern Rockies'
    },
    'Southern_Rockies': {
        'lat': (36.5, 41.5), 'lon': (-109.0, -104.5),
        'color': '#d62728', 'label': 'Southern Rockies (CO)'
    },
    # Idaho: stesso bbox usato per sviluppare e testare la U-Net originale
    # (lon -118.5/-109.5, lat 41/50) ritagliato al dominio SPORTLIS esteso
    'Idaho': {
        'lat': (42.0, 49.0), 'lon': (-117.5, -111.0),
        'color': '#9467bd', 'label': 'Idaho (dominio originale)'
    },
}

def make_basin_mask(lat_1d, lon_1d, lat_range, lon_range):
    """Maschera booleana (H, W) per un bacino rettangolare."""
    lat2d, lon2d = np.meshgrid(lat_1d, lon_1d, indexing='ij')
    return ((lat2d >= lat_range[0]) & (lat2d <= lat_range[1]) &
            (lon2d >= lon_range[0]) & (lon2d <= lon_range[1]))

basin_masks = {}
for name, info in BASINS.items():
    mask = make_basin_mask(lat_1d, lon_1d, info['lat'], info['lon'])
    basin_masks[name] = mask
    print(f"{info['label']:30s}: {mask.sum():6d} pixel ({100*mask.sum()/(EXT_H*EXT_W):.1f}% del dominio)")

# ── Mappa dei bacini ─────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 7))
basin_map = np.zeros((EXT_H, EXT_W))
for i, (name, mask) in enumerate(basin_masks.items(), 1):
    basin_map[mask] = i

colors_list = ['white'] + [BASINS[n]['color'] for n in basin_masks]
from matplotlib.colors import ListedColormap
cmap = ListedColormap(colors_list)
ax.imshow(basin_map, origin='lower', cmap=cmap,
          extent=[EXT_LON_MIN, EXT_LON_MAX, EXT_LAT_MIN, EXT_LAT_MAX],
          aspect='auto', alpha=0.7, vmin=0, vmax=len(BASINS))
patches = [mpatches.Patch(color=info['color'], label=info['label'])
           for info in BASINS.values()]
ax.legend(handles=patches, loc='lower right', fontsize=9)
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
ax.set_title('Bacini di analisi nel dominio SPORTLIS esteso')
plt.tight_layout()
plt.savefig(FIG_DIR / 'basin_map.png', dpi=150, bbox_inches='tight')
plt.show()
print('Salvato: basin_map.png')

In [ ]:
# ── XAI Barplot ───────────────────────────────────────────────────────────────
xai_npz = OUTPUT_DIR / 'xai_extended_importance.npz'
xai_csv = OUTPUT_DIR / 'xai_extended_importance.csv'

if not xai_npz.exists():
    print(f'File non trovato: {xai_npz}')
else:
    xai = np.load(str(xai_npz), allow_pickle=True)
    imp_matrix   = xai['imp_matrix']        # (n_folds, n_channels)
    channel_names = list(xai['channel_names'])
    channel_group = list(xai['channel_group'])
    mae_baseline  = xai['mae_baseline']

    imp_mean = np.nanmean(imp_matrix, axis=0)
    imp_std  = np.nanstd(imp_matrix,  axis=0)
    order    = np.argsort(-imp_mean)

    gc_color = {'temporal': '#d32f2f', 'topo': '#388e3c', 'coord': '#616161'}
    names_s  = [channel_names[i] for i in order]
    grps_s   = [channel_group[i] for i in order]
    mean_s   = imp_mean[order]
    std_s    = imp_std[order]
    colors   = [gc_color.get(g, 'grey') for g in grps_s]

    fig, ax = plt.subplots(figsize=(10, 7))
    yp = np.arange(len(names_s))
    ax.barh(yp, mean_s, xerr=std_s, color=colors, edgecolor='k',
            alpha=0.85, capsize=3, height=0.7)
    ax.set_yticks(yp); ax.set_yticklabels(names_s, fontsize=10)
    ax.axvline(0, color='k', lw=0.8, ls='--')
    ax.set_xlabel('Importanza permutazione (ΔMAE mm)', fontsize=11)
    ax.set_title('XAI Permutation Importance — U-Net SWE Extended (1985–2024)', fontsize=12)
    patches = [mpatches.Patch(color=c, label=l)
               for c,l in [('#d32f2f','Temporale'),('#388e3c','Topografico'),('#616161','Coordinate')]]
    ax.legend(handles=patches, fontsize=9)
    ax.invert_yaxis()
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'xai_barplot.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Salvato: xai_barplot.png')

    print(f'\nBaseline MAE medio: {np.nanmean(mae_baseline):.2f} mm')
    print(f'{"Rank":>4}  {"Channel":<18s} {"Group":<10s}  {"mean":>8s}  {"std":>6s}')
    for r,i in enumerate(order,1):
        print(f'  {r:>2d}  {channel_names[i]:<18s} {channel_group[i]:<10s}  '
              f'{imp_mean[i]:>+8.3f}  {imp_std[i]:>6.3f}')

In [ ]:
# ── XAI Permutation Importance per Bacino ────────────────────────────────────
# Per ogni fold LOYO campiona N_SAMPLE timestep snowy, esegue l'inference una
# volta (baseline) e poi permuta un canale alla volta (temporalmente) e riesegue.
# ΔMAE = MAE_permutato - MAE_baseline per ogni bacino+canale.
# N_SAMPLE=200 ≈ 17 pass × 200 step × 8 fold → ~3-4 min su V100.
# Resume: salva dopo ogni fold in xai_basin_importance.npz.

N_SAMPLE      = 200
BATCH_SIZE    = 32
SWE_MAX_MM    = 3000.0
XAI_BASIN_NPZ = OUTPUT_DIR / 'xai_basin_importance.npz'

basin_names   = list(BASINS.keys())
n_basins      = len(basin_names)
n_folds       = len(LOYO_FOLDS)
n_ch          = N_IN_CHANNELS

# ── Resume ────────────────────────────────────────────────────────────────────
if XAI_BASIN_NPZ.exists():
    _z = np.load(str(XAI_BASIN_NPZ), allow_pickle=True)
    imp_basin  = _z['imp_basin']     # (n_basins, n_folds, n_ch)
    mae_base_all = _z['mae_base_all'] # (n_basins, n_folds)
    done_folds = list(_z['done_folds'])
    print(f'Resume: fold già completati = {done_folds}')
else:
    imp_basin    = np.full((n_basins, n_folds, n_ch), np.nan, dtype=np.float32)
    mae_base_all = np.full((n_basins, n_folds),       np.nan, dtype=np.float32)
    done_folds   = []

lat2d, lon2d = np.meshgrid(lat_1d, lon_1d, indexing='ij')
lat_n = ((lat2d-lat2d.mean())/(lat2d.std() or 1.0)).astype(np.float32)
lon_n = ((lon2d-lon2d.mean())/(lon2d.std() or 1.0)).astype(np.float32)

def _infer(model, X):
    """(N,16,H,W) → pred mm (N,H,W). Chunk su BATCH_SIZE."""
    out = np.empty((len(X), EXT_H, EXT_W), dtype=np.float32)
    with torch.no_grad():
        for b0 in range(0, len(X), BATCH_SIZE):
            b1 = min(b0+BATCH_SIZE, len(X))
            out[b0:b1] = (model(torch.from_numpy(X[b0:b1]).to(device))
                          .cpu().numpy()[:,0])
    return np.expm1(np.clip(out, 0, np.log1p(SWE_MAX_MM)))

for fi, fold in enumerate(LOYO_FOLDS):
    ty = fold['test']
    if ty in done_folds:
        print(f'Fold {ty}: skip'); continue

    mm_path  = MEMMAP_DIR / f'y{ty}_feat.npy'
    msk_path = MEMMAP_DIR / f'y{ty}_mask.npy'
    ckpt     = OUTPUT_DIR  / CHECKPOINT_TEMPLATE.format(test=ty)
    if not ckpt.exists() or not mm_path.exists():
        print(f'Fold {ty}: SKIP file mancanti'); continue

    print(f'\nFold {ty}:', end=' ', flush=True)
    mean_d, std_d = load_stats(ty)
    mm  = np.lib.format.open_memmap(str(mm_path),  mode='r')
    msk = np.lib.format.open_memmap(str(msk_path), mode='r')
    T = mm.shape[0]; n_feat = len(INPUT_VARS)

    # ── Campiona N_SAMPLE timestep snowy ─────────────────────────────────────
    rng = np.random.default_rng(seed=ty)
    idx = np.sort(rng.choice(T, size=min(N_SAMPLE, T), replace=False))
    ns  = len(idx)

    # ── Costruisci X campionato ───────────────────────────────────────────────
    X_s = np.empty((ns, n_ch, EXT_H, EXT_W), dtype=np.float32)
    for i, v in enumerate(INPUT_VARS):
        X_s[:,i] = (mm[idx,i].astype(np.float32) - mean_d[v]) / max(std_d[v], 1e-6)
    X_s[:,n_feat]    = lat_n[None]
    X_s[:,n_feat+1]  = lon_n[None]
    X_s[:,n_feat+2:] = topo_tensor[None]
    obs_s = np.clip(mm[idx, n_feat].astype(np.float32), 0, SWE_MAX_MM)
    msk_s = (msk[idx] if msk.ndim==3 else msk[idx,0]).astype(bool)
    del mm, msk; gc.collect()

    model = UNet(N_IN_CHANNELS, dropout=DROPOUT_P).to(device)
    model.load_state_dict(torch.load(str(ckpt), map_location=device))
    model.eval()

    # ── Baseline ──────────────────────────────────────────────────────────────
    pred_base = _infer(model, X_s)
    for bi, bname in enumerate(basin_names):
        valid = msk_s & basin_masks[bname][None]
        if valid.any():
            mae_base_all[bi, fi] = float(np.abs(pred_base[valid]-obs_s[valid]).mean())
    print(f'base MAE/bacino: {[f"{mae_base_all[bi,fi]:.2f}" for bi in range(n_basins)]}')

    # ── Permutazione canale per canale ────────────────────────────────────────
    for ci in range(n_ch):
        X_p = X_s.copy()
        X_p[:,ci] = X_s[rng.permutation(ns), ci]  # permuta asse tempo
        pred_p = _infer(model, X_p); del X_p
        for bi, bname in enumerate(basin_names):
            valid = msk_s & basin_masks[bname][None]
            if valid.any():
                mae_p = float(np.abs(pred_p[valid]-obs_s[valid]).mean())
                imp_basin[bi, fi, ci] = mae_p - mae_base_all[bi, fi]
        del pred_p; gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        print(f'  ch{ci:2d} {CHANNEL_NAMES[ci]:<20s}', end='', flush=True)
        print(f' ΔMAE: {[f"{imp_basin[bi,fi,ci]:+.3f}" for bi in range(n_basins)]}')

    del model, pred_base, X_s, obs_s, msk_s; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    done_folds.append(ty)
    np.savez(str(XAI_BASIN_NPZ),
             imp_basin=imp_basin, mae_base_all=mae_base_all,
             done_folds=np.array(done_folds),
             basin_names=np.array(basin_names),
             channel_names=np.array(CHANNEL_NAMES),
             channel_group=np.array(CHANNEL_GROUP))
    print(f'  → Salvato (fold {ty})')

print(f'\nXAI per bacino completato. Folds: {done_folds}')
print(f'Shape imp_basin: {imp_basin.shape}  (bacini × fold × canali)')


In [ ]:
# ── Plot XAI per Bacino ────────────────────────────────────────────────────────
# Doppia visualizzazione:
#  1. Heatmap (bacino × canale): ΔMAE medio normalizzato per baseline MAE
#  2. Barplot per ogni bacino: top canali per importanza

if not XAI_BASIN_NPZ.exists():
    print('Esegui prima la cella XAI per bacino.')
else:
    _z = np.load(str(XAI_BASIN_NPZ), allow_pickle=True)
    imp_b   = _z['imp_basin']      # (n_basins, n_folds, n_ch)
    mae_b   = _z['mae_base_all']   # (n_basins, n_folds)
    bnames  = list(_z['basin_names'])
    cnames  = list(_z['channel_names'])
    cgroups = list(_z['channel_group'])
    n_basins, n_folds, n_ch = imp_b.shape

    # ΔMAE medio tra fold, normalizzato per baseline MAE di quel bacino
    imp_mean = np.nanmean(imp_b, axis=1)                        # (n_basins, n_ch)
    base_mean = np.nanmean(mae_b, axis=1, keepdims=True)        # (n_basins, 1)
    imp_rel   = imp_mean / np.where(base_mean>0, base_mean, 1)  # ΔMAE relativo

    gc_color = {'temporal': '#d32f2f', 'topo': '#388e3c', 'coord': '#616161'}
    basin_labels = [BASINS[n]['label'] for n in bnames]

    # ── 1. Heatmap ────────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(14, 5))
    im = ax.imshow(imp_rel, aspect='auto', cmap='RdYlGn_r',
                   vmin=0, vmax=np.nanpercentile(imp_rel, 95))
    ax.set_xticks(range(n_ch))
    ax.set_xticklabels(cnames, rotation=45, ha='right', fontsize=8)
    ax.set_yticks(range(n_basins))
    ax.set_yticklabels(basin_labels, fontsize=10)
    # Colora etichette canali per gruppo
    for tick, name in zip(ax.get_xticklabels(), cnames):
        g = cgroups[cnames.index(name)]
        tick.set_color(gc_color.get(g, 'k'))
    # Valori nelle celle
    for bi in range(n_basins):
        for ci in range(n_ch):
            v = imp_rel[bi, ci]
            if not np.isnan(v):
                ax.text(ci, bi, f'{v:.2f}', ha='center', va='center',
                        fontsize=6, color='white' if v>0.3 else 'k')
    plt.colorbar(im, ax=ax, label='ΔMAE / MAE_baseline (importanza relativa)')
    ax.set_title('XAI Permutation Importance per Bacino — ΔMAE relativo (medio su fold LOYO)',
                 fontsize=12)
    patches = [mpatches.Patch(color=c, label=l)
               for c,l in [('#d32f2f','Temporale'),('#388e3c','Topografico'),('#616161','Coordinate')]]
    ax.legend(handles=patches, fontsize=8, loc='upper right')
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'xai_basin_heatmap.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Salvato: xai_basin_heatmap.png')

    # ── 2. Barplot per bacino (top-5 canali) ─────────────────────────────────
    n_top = 8
    n_rows = (n_basins + 2) // 3
    fig, axes = plt.subplots(n_rows, 3, figsize=(16, 4*n_rows))
    axes = axes.flatten()

    for bi, (bname, ax) in enumerate(zip(bnames, axes)):
        order = np.argsort(-imp_mean[bi])[:n_top]
        vals  = imp_mean[bi][order]
        lbls  = [cnames[i] for i in order]
        grps  = [cgroups[i] for i in order]
        cols  = [gc_color.get(g, 'grey') for g in grps]
        yp    = np.arange(len(order))
        # std tra fold
        std_vals = np.nanstd(imp_b[:,: , order], axis=1)  # oops, need per channel
        # compute properly
        std_v = np.array([np.nanstd(imp_b[bi,:,i]) for i in order])
        ax.barh(yp, vals, xerr=std_v, color=cols, edgecolor='k',
                alpha=0.85, capsize=3, height=0.7)
        ax.set_yticks(yp); ax.set_yticklabels(lbls, fontsize=8)
        ax.axvline(0, color='k', lw=0.8, ls='--')
        ax.set_title(BASINS[bname]['label'], fontsize=10, fontweight='bold',
                     color=BASINS[bname]['color'])
        ax.set_xlabel('ΔMAE (mm)', fontsize=8)
        ax.invert_yaxis()

    for ax in axes[n_basins:]:
        ax.set_visible(False)

    patches = [mpatches.Patch(color=c, label=l)
               for c,l in [('#d32f2f','Temporale'),('#388e3c','Topografico'),('#616161','Coordinate')]]
    fig.legend(handles=patches, fontsize=9, loc='lower right')
    plt.suptitle('XAI per Bacino — Top-8 canali per importanza permutazione',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'xai_basin_barplots.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Salvato: xai_basin_barplots.png')

    # ── Tabella riassuntiva: feature #1 per bacino ────────────────────────────
    print(f'\n{"Bacino":<30s}  {"Feature #1":<22s}  {"ΔMAE":>8s}  {"Base MAE":>9s}')
    print('-'*75)
    for bi, bname in enumerate(bnames):
        top = int(np.nanargmax(imp_mean[bi]))
        print(f'{BASINS[bname]["label"]:<30s}  {cnames[top]:<22s}  ')
        print(f'  {imp_mean[bi,top]:>+8.3f}  {base_mean[bi,0]:>9.3f}', end='')
        # Feature #1 topo vs temporale
        g = cgroups[top]
        print(f'  [{g}]')


In [ ]:
# ── Performance per bacino sui fold LOYO ──────────────────────────────────────
# Carica U-Net (identica all'architettura di training)
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch, dropout=0.0):
        super().__init__()
        g = min(8, out_ch)
        while out_ch % g != 0: g -= 1
        layers = [nn.Conv2d(in_ch,out_ch,3,padding=1), nn.GroupNorm(g,out_ch), nn.GELU(),
                  nn.Conv2d(out_ch,out_ch,3,padding=1), nn.GroupNorm(g,out_ch), nn.GELU()]
        if dropout > 0: layers.append(nn.Dropout2d(dropout))
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x)

def pad_to_mult(x, m=8):
    h,w = x.shape[-2],x.shape[-1]
    H = ((h+m-1)//m)*m; W = ((w+m-1)//m)*m
    ph = H-h; pw = W-w
    pl = pw//2; pr = pw-pl; pt = ph//2; pb = ph-pt
    return F.pad(x,(pl,pr,pt,pb),mode='reflect'),(pl,pr,pt,pb)

def unpad(x, p):
    pl,pr,pt,pb = p; H,W = x.shape[-2],x.shape[-1]
    return x[..., pt:H-pb if pb>0 else H, pl:W-pr if pr>0 else W]

class UNet(nn.Module):
    def __init__(self, in_channels, out_channels=1, base=32, dropout=0.1):
        super().__init__()
        self.e1=DoubleConv(in_channels,base); self.p1=nn.MaxPool2d(2)
        self.e2=DoubleConv(base,base*2);      self.p2=nn.MaxPool2d(2)
        self.e3=DoubleConv(base*2,base*4);    self.p3=nn.MaxPool2d(2)
        self.bn=DoubleConv(base*4,base*8,dropout=dropout)
        self.u1=nn.ConvTranspose2d(base*8,base*4,2,stride=2); self.d1=DoubleConv(base*8,base*4)
        self.u2=nn.ConvTranspose2d(base*4,base*2,2,stride=2); self.d2=DoubleConv(base*4,base*2)
        self.u3=nn.ConvTranspose2d(base*2,base,2,stride=2);   self.d3=DoubleConv(base*2,base)
        self.out=nn.Conv2d(base,out_channels,1)
    def forward(self,x):
        x,p=pad_to_mult(x); e1=self.e1(x); e2=self.e2(self.p1(e1))
        e3=self.e3(self.p2(e2)); bn=self.bn(self.p3(e3))
        d1=self.d1(torch.cat([self.u1(bn),e3],1))
        d2=self.d2(torch.cat([self.u2(d1),e2],1))
        d3=self.d3(torch.cat([self.u3(d2),e1],1))
        return unpad(self.out(d3),p)

print('Architettura U-Net definita.')

In [ ]:
# ── Carica topo_tensor da AUX_DIR ────────────────────────────────────────────
STATIC_FILE = AUX_DIR / 'sportlis_static_extended.nc'
CANOPY_FILE = AUX_DIR / 'sportlis_canopy_extended_3km.nc'
TPI_RADII   = {'tpi_short': 5, 'tpi_long': 10}

ds_static = xr.open_dataset(STATIC_FILE)
static_dict = {v: ds_static[v].values.astype(np.float32)
               for v in ['elevation','slope','aspect_sin','aspect_cos']}
elev = static_dict['elevation']
elev_filled = np.where(np.isfinite(elev), elev, np.nanmedian(elev)).astype(np.float32)

tpi_layers = {}
for name, r in TPI_RADII.items():
    loc_mean = uniform_filter(elev_filled, size=2*r+1, mode='nearest').astype(np.float32)
    tpi_layers[name] = (elev_filled - loc_mean).astype(np.float32)

ds_canopy = xr.open_dataset(CANOPY_FILE)
canopy_fraction = ds_canopy['canopy_fraction'].values.astype(np.float32)

topo_layers_raw = {}
for v in ['elevation','slope','aspect_sin','aspect_cos']:
    arr = static_dict[v].astype(np.float32)
    if v in ('aspect_sin','aspect_cos'): arr = np.where(np.isfinite(arr), arr, 0.0)
    else: arr = np.where(np.isnan(arr), np.nanmedian(arr), arr)
    topo_layers_raw[v] = arr
for name, arr in tpi_layers.items():
    topo_layers_raw[name] = arr
topo_layers_raw['canopy_fraction'] = canopy_fraction

topo_normalized = []
for name in TOPO_VARS:
    arr = topo_layers_raw[name]
    if name in ('aspect_sin','aspect_cos'):
        topo_normalized.append(arr)
    elif name == 'canopy_fraction':
        topo_normalized.append((arr - float(np.nanmean(arr))).astype(np.float32))
    else:
        m = float(np.nanmean(arr)); s = float(np.nanstd(arr)) or 1.0
        topo_normalized.append(((arr-m)/s).astype(np.float32))

topo_tensor = np.stack(topo_normalized, axis=0).astype(np.float32)
print(f'topo_tensor: {topo_tensor.shape}')

In [ ]:
# ── Funzione inference per un anno ────────────────────────────────────────────
# Legge dalle memmap pre-baked (T_snowy, N_FEAT+1, H, W):
#   canali 0..N_FEAT-1 = INPUT_VARS (già calcolati), canale N_FEAT = target log1p(SWE)
# Le timestamps vengono lette dallo zarr (solo 'time' + 'is_snowy_time').

def open_zarr_robust(path):
    p = str(path)
    for kw in [dict(consolidated=True), dict(consolidated=False),
               dict(consolidated=True, zarr_format=3),
               dict(consolidated=False, zarr_format=3)]:
        try: return xr.open_zarr(p, chunks={}, **kw)
        except: continue
    return xr.open_zarr(p, chunks={})

def zarr_path(year):
    return ZARR_DIR / f'sportlis_narr_pp_{year}.zarr'

def get_closest_fold(year):
    test_years = [f['test'] for f in LOYO_FOLDS]
    if year in test_years:
        return year
    return min(test_years, key=lambda ty: abs(ty - year))

def load_stats(test_year):
    sc = np.load(str(OUTPUT_DIR / f'norm_stats_test{test_year}.npz'), allow_pickle=True)
    mean_arr = sc['mean_arr']; std_arr = sc['std_arr']
    return ({v: float(mean_arr[i]) for i,v in enumerate(INPUT_VARS)},
            {v: float(std_arr[i])  for i,v in enumerate(INPUT_VARS)})

def predict_year(year, fold_test_year=None, batch_size=4):
    """Inference per un anno intero.
    Legge feature temporali dai memmap pre-baked (evita KeyError zarr).
    Restituisce (preds_mm, swe_obs_mm, times, mask_snowy).
    """
    if fold_test_year is None:
        fold_test_year = get_closest_fold(year)

    ckpt_path = OUTPUT_DIR / CHECKPOINT_TEMPLATE.format(test=fold_test_year)
    if not ckpt_path.exists():
        print(f'  Checkpoint non trovato: {ckpt_path}')
        return None, None, None, None

    mean_d, std_d = load_stats(fold_test_year)

    mm_path  = MEMMAP_DIR / f'y{year}_feat.npy'
    msk_path = MEMMAP_DIR / f'y{year}_mask.npy'

    if not mm_path.exists():
        print(f'  Memmap non trovato: {mm_path} — build memmap prima di eseguire inference.')
        return None, None, None, None

    # ── Carica memmap ────────────────────────────────────────────────────────
    mm  = np.lib.format.open_memmap(str(mm_path),  mode='r')
    msk = np.lib.format.open_memmap(str(msk_path), mode='r')
    T_snowy = mm.shape[0]
    n_feat  = len(INPUT_VARS)  # 7

    feat_raw = mm[:, :n_feat].copy().astype(np.float32)   # (T_snowy, 7, H, W)
    swe_raw  = mm[:, n_feat ].copy().astype(np.float32)   # (T_snowy, H, W)
    mask_snowy = (msk[:, 0] if msk.ndim == 4 else msk[:]).copy().astype(np.float32)

    # ── Normalizza feature temporali ─────────────────────────────────────────
    for i, v in enumerate(INPUT_VARS):
        feat_raw[:, i] = (feat_raw[:, i] - mean_d[v]) / max(std_d[v], 1e-6)

    # ── Coordinate + topo ────────────────────────────────────────────────────
    lat2d, lon2d = np.meshgrid(lat_1d, lon_1d, indexing='ij')
    lat_norm = ((lat2d - lat2d.mean()) / (lat2d.std() or 1.0)).astype(np.float32)
    lon_norm = ((lon2d - lon2d.mean()) / (lon2d.std() or 1.0)).astype(np.float32)
    lat_t = np.broadcast_to(lat_norm[None], (T_snowy, EXT_H, EXT_W)).copy()[:, None]
    lon_t = np.broadcast_to(lon_norm[None], (T_snowy, EXT_H, EXT_W)).copy()[:, None]
    topo_t = np.broadcast_to(topo_tensor[None], (T_snowy, 7, EXT_H, EXT_W)).copy()

    X = np.concatenate([feat_raw, lat_t, lon_t, topo_t], axis=1).astype(np.float32)
    del feat_raw, lat_t, lon_t, topo_t; gc.collect()

    # ── Timestamps da zarr (solo time + is_snowy_time) ───────────────────────
    try:
        ds = open_zarr_robust(zarr_path(year))
        all_times = pd.to_datetime(ds['time'].values)
        if 'is_snowy_time' in ds:
            snowy_idx = np.where(ds['is_snowy_time'].values)[0]
            times = all_times[snowy_idx] if len(snowy_idx) == T_snowy else all_times[:T_snowy]
        else:
            times = all_times[:T_snowy]
        ds.close()
    except Exception as e:
        print(f'  Zarr non disponibile ({e}), times sintetiche')
        times = pd.date_range(f'{year}-01-01', periods=T_snowy, freq='3h')

    # ── Inference ────────────────────────────────────────────────────────────
    model = UNet(N_IN_CHANNELS, dropout=DROPOUT_P).to(device)
    model.load_state_dict(torch.load(str(ckpt_path), map_location=device))
    model.eval()

    preds = np.zeros((T_snowy, EXT_H, EXT_W), dtype=np.float32)
    with torch.no_grad():
        for t0 in range(0, T_snowy, batch_size):
            t1 = min(t0 + batch_size, T_snowy)
            xb = torch.from_numpy(X[t0:t1]).to(device)
            pb = model(xb).cpu().numpy()[:, 0]
            preds[t0:t1] = np.expm1(np.clip(pb, 0, np.log1p(SWE_MAX_MM)))

    SWE_MAX_MM = 3000.0  # clip artefatti IDW sopra massimo fisico ragionevole
    swe_obs = np.clip(swe_raw, 0.0, SWE_MAX_MM)

    del model, mm, msk, X; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

    print(f'  {year}: T_snowy={T_snowy}  '
          f'pred_mean={preds[mask_snowy > 0].mean():.1f} mm  '
          f'obs_mean={swe_obs[mask_snowy > 0].mean():.1f} mm')
    return preds, swe_obs, times, mask_snowy

print('Funzioni inference definite (lettura da memmap).')


In [ ]:
# ── DIAGNOSTICA: verifica stats + output modello ──────────────────────────────
# Eseguire PRIMA delle celle di analisi per verificare che tutto sia corretto.

import os
_ty = 1985  # fold test da diagnosticare

# ── 1. Norm stats ─────────────────────────────────────────────────────────────
stats_file = OUTPUT_DIR / f'norm_stats_test{_ty}.npz'
if not stats_file.exists():
    print(f'Stats non trovate: {stats_file}')
else:
    mean_d, std_d = load_stats(_ty)
    print('=== NORM STATS ===')
    for v in INPUT_VARS:
        print(f'  {v:<20s}: mean={mean_d[v]:>10.4f}  std={std_d[v]:>10.4f}')

# ── 2. Contenuto memmap ───────────────────────────────────────────────────────
mm_p = MEMMAP_DIR / f'y{_ty}_feat.npy'
if mm_p.exists():
    mm = np.lib.format.open_memmap(str(mm_p), mode='r')
    print(f'\n=== MEMMAP SHAPE: {mm.shape} ===')
    n_feat = len(INPUT_VARS)
    for i, v in enumerate(INPUT_VARS):
        a = mm[:, i]
        print(f'  ch{i} {v:<20s}: min={a.min():.3f}  max={a.max():.3f}  mean={a.mean():.3f}')
    swe_ch = mm[:, n_feat]
    print(f'  ch{n_feat} swe_target_filled   : min={swe_ch.min():.3f}  '
          f'max={swe_ch.max():.3f}  mean={swe_ch.mean():.3f}  '
          f'p95={np.percentile(swe_ch[swe_ch>0], 95):.3f}')
    print(f'  -> expm1(clip(swe_ch,0,10)) range: [{np.expm1(np.clip(swe_ch.min(),0,10)):.1f}, '
          f'{np.expm1(np.clip(swe_ch.max(),0,10)):.1f}]')
    del mm

# ── 3. Output modello (1 timestep) ────────────────────────────────────────────
ckpt_p = OUTPUT_DIR / CHECKPOINT_TEMPLATE.format(test=_ty)
if ckpt_p.exists() and mm_p.exists():
    mm = np.lib.format.open_memmap(str(mm_p), mode='r')
    msk = np.lib.format.open_memmap(str(MEMMAP_DIR / f'y{_ty}_mask.npy'), mode='r')
    n_feat = len(INPUT_VARS)
    mean_d, std_d = load_stats(_ty)
    
    f0 = mm[0:1, :n_feat].copy().astype(np.float32)
    for i, v in enumerate(INPUT_VARS):
        f0[:, i] = (f0[:, i] - mean_d[v]) / max(std_d[v], 1e-6)
    
    lat2d, lon2d = np.meshgrid(lat_1d, lon_1d, indexing='ij')
    lat_n = ((lat2d - lat2d.mean()) / (lat2d.std() or 1.0)).astype(np.float32)
    lon_n = ((lon2d - lon2d.mean()) / (lon2d.std() or 1.0)).astype(np.float32)
    X = np.concatenate([f0, lat_n[None, None], lon_n[None, None],
                        topo_tensor[None]], axis=1).astype(np.float32)
    
    print(f'\n=== INPUT TENSOR: shape={X.shape}  min={X.min():.3f}  max={X.max():.3f} ===')
    
    model = UNet(N_IN_CHANNELS, dropout=DROPOUT_P).to(device)
    model.load_state_dict(torch.load(str(ckpt_p), map_location=device))
    model.eval()
    with torch.no_grad():
        pb = model(torch.from_numpy(X).to(device)).cpu().numpy()[0, 0]
    
    mask0 = (msk[0] if msk.ndim == 3 else msk[0, 0]).astype(bool)
    print(f'\n=== MODELLO OUTPUT (log1p space) ===')
    print(f'  ALL pixels: min={pb.min():.4f}  max={pb.max():.4f}  mean={pb.mean():.4f}')
    if mask0.any():
        print(f'  SNOWY px:   min={pb[mask0].min():.4f}  max={pb[mask0].max():.4f}  '
              f'mean={pb[mask0].mean():.4f}')
    print(f'  % > 0: {100*(pb>0).mean():.1f}%   % > 1: {100*(pb>1).mean():.1f}%')
    pred_mm = np.expm1(np.clip(pb, 0, 10))
    print(f'\n=== PREDIZIONE SWE (mm) ===')
    print(f'  mean={pred_mm.mean():.2f}  snowy mean={pred_mm[mask0].mean():.2f}  max={pred_mm.max():.2f}')
    
    # atteso
    swe_raw = mm[0, n_feat]
    swe_obs = np.maximum(swe_raw, 0.0)
    print(f'\n=== SWE OSSERVATO (raw mm) ===')
    print(f'  mean all={swe_obs.mean():.2f}  snowy={swe_obs[mask0].mean():.2f}  max={swe_obs.max():.2f}')
    
    del model, mm, msk; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

print('\nDiagnostica completata.')


In [ ]:
# ── Time series per bacino — anni di test (out-of-sample) ─────────────────────
# Chunk-based inference. Resume per (anno, bacino): se un bacino è nuovo viene
# calcolato senza rieseguire gli altri.

TEST_YEARS   = [f['test'] for f in LOYO_FOLDS]   # [1985,1990,...,2020]
TIME_CHUNK   = 32
BATCH_SIZE   = 32
SWE_MAX_MM   = 3000.0
BASIN_TS_CSV = OUTPUT_DIR / 'basin_ts_results.csv'

# ── Carica risultati parziali (resume per anno+bacino) ───────────────────────
import csv
done_keys = set()   # (year, basin_name)
if BASIN_TS_CSV.exists():
    _df = pd.read_csv(BASIN_TS_CSV)
    for _, row in _df.iterrows():
        done_keys.add((int(row['year']), str(row['basin'])))
    print(f'Resume: {len(done_keys)} combinazioni (anno, bacino) già presenti')
    missing = [(ty, n) for ty in TEST_YEARS for n in BASINS if (ty,n) not in done_keys]
    print(f'  Mancanti: {len(missing)} — {missing[:6]}{"..." if len(missing)>6 else ""}')
else:
    missing = [(ty, n) for ty in TEST_YEARS for n in BASINS]
    print(f'Nuovo CSV: calcolo {len(missing)} combinazioni')

# Raggruppa per anno (una sola passata inference per anno)
from itertools import groupby
years_needed = sorted(set(ty for ty, _ in missing))
basins_by_year = {ty: [n for yy,n in missing if yy==ty] for ty in years_needed}

for ty in years_needed:
    basins_todo = basins_by_year[ty]
    print(f'\nAnno test={ty} — calcolo bacini: {basins_todo}')

    ckpt_path = OUTPUT_DIR / CHECKPOINT_TEMPLATE.format(test=ty)
    mm_path   = MEMMAP_DIR / f'y{ty}_feat.npy'
    msk_path  = MEMMAP_DIR / f'y{ty}_mask.npy'
    if not ckpt_path.exists() or not mm_path.exists():
        print(f'  SKIP: file mancanti'); continue

    mean_d, std_d = load_stats(ty)
    mm  = np.lib.format.open_memmap(str(mm_path),  mode='r')
    msk = np.lib.format.open_memmap(str(msk_path), mode='r')
    T_snowy = mm.shape[0]; n_feat = len(INPUT_VARS)

    lat2d, lon2d = np.meshgrid(lat_1d, lon_1d, indexing='ij')
    lat_n = ((lat2d-lat2d.mean())/(lat2d.std() or 1.0)).astype(np.float32)
    lon_n = ((lon2d-lon2d.mean())/(lon2d.std() or 1.0)).astype(np.float32)

    model = UNet(N_IN_CHANNELS, dropout=DROPOUT_P).to(device)
    model.load_state_dict(torch.load(str(ckpt_path), map_location=device))
    model.eval()

    bacc = {name: {'sp':0.0,'so':0.0,'sae':0.0,'se':0.0,'n':0} for name in basins_todo}

    for tc0 in range(0, T_snowy, TIME_CHUNK):
        tc1 = min(tc0+TIME_CHUNK, T_snowy); tc = tc1-tc0
        X = np.empty((tc, N_IN_CHANNELS, EXT_H, EXT_W), dtype=np.float32)
        for i, v in enumerate(INPUT_VARS):
            X[:,i] = (mm[tc0:tc1,i].astype(np.float32)-mean_d[v])/max(std_d[v],1e-6)
        X[:,n_feat]=lat_n[None]; X[:,n_feat+1]=lon_n[None]; X[:,n_feat+2:]=topo_tensor[None]
        pb_chunk = np.empty((tc, EXT_H, EXT_W), dtype=np.float32)
        with torch.no_grad():
            for b0 in range(0, tc, BATCH_SIZE):
                b1 = min(b0+BATCH_SIZE, tc)
                pb_chunk[b0:b1] = model(torch.from_numpy(X[b0:b1]).to(device)).cpu().numpy()[:,0]
        del X
        pb_mm  = np.expm1(np.clip(pb_chunk,0,np.log1p(SWE_MAX_MM))); del pb_chunk
        obs_mm = np.clip(mm[tc0:tc1,n_feat].astype(np.float32), 0, SWE_MAX_MM)
        msk_c  = (msk[tc0:tc1] if msk.ndim==3 else msk[tc0:tc1,0]).astype(bool)
        for name in basins_todo:
            valid = msk_c & basin_masks[name][None]
            if not valid.any(): continue
            b = bacc[name]; pv=pb_mm[valid]; ov=obs_mm[valid]
            b['sp']+=float(pv.sum()); b['so']+=float(ov.sum())
            b['sae']+=float(np.abs(pv-ov).sum()); b['se']+=float((pv-ov).sum())
            b['n']+=int(valid.sum())
        del pb_mm, obs_mm, msk_c; gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

    del model, mm, msk; gc.collect()

    rows = []
    for name in basins_todo:
        b=bacc[name]; n=max(b['n'],1)
        row=dict(year=ty,basin=name,pred_mean=b['sp']/n,obs_mean=b['so']/n,
                 mae=b['sae']/n,bias=b['se']/n,n_pixels=b['n'])
        rows.append(row)
        print(f"  {BASINS[name]['label']:30s}: pred={b['sp']/n:6.1f}  obs={b['so']/n:6.1f}  "
              f"mae={b['sae']/n:6.1f}  n={b['n']:,}")

    write_header = not BASIN_TS_CSV.exists()
    with open(BASIN_TS_CSV,'a',newline='') as f:
        w=csv.DictWriter(f,fieldnames=rows[0].keys())
        if write_header: w.writeheader()
        w.writerows(rows)
    print(f'  Salvato → {BASIN_TS_CSV.name}')

# ── Ricostruisce basin_ts da CSV completo ─────────────────────────────────────
_df = pd.read_csv(BASIN_TS_CSV)
basin_ts = {name:{'year':[],'pred_mean':[],'obs_mean':[],'mae':[],'bias':[]}
            for name in BASINS}
for _, row in _df.sort_values('year').iterrows():
    name=row['basin']
    if name not in basin_ts: continue
    basin_ts[name]['year'].append(int(row['year']))
    for k in ['pred_mean','obs_mean','mae','bias']: basin_ts[name][k].append(float(row[k]))

# ── Diagnostica Idaho ─────────────────────────────────────────────────────────
id_ts = basin_ts['Idaho']
print(f'\nIdaho pixels: {_df[_df.basin=="Idaho"]["n_pixels"].tolist()}')
print(f'Idaho obs_mean: {[round(v,2) for v in id_ts["obs_mean"]]}')
print(f'Idaho pred_mean: {[round(v,2) for v in id_ts["pred_mean"]]}')
print(f'\nDone. Anni in CSV: {sorted(_df["year"].unique().tolist())}')


In [ ]:
# ── Plot: MAE per bacino nel tempo ────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharey=False)
axes = axes.flatten()

for ax, (name, info) in zip(axes, BASINS.items()):
    ts = basin_ts[name]
    ax.bar(ts['year'], ts['mae'], color=info['color'], alpha=0.8, width=3)
    ax.set_title(info['label'], fontsize=10)
    ax.set_xlabel('Anno test'); ax.set_ylabel('MAE (mm)')
    ax.set_xticks(ts['year']); ax.set_xticklabels(ts['year'], rotation=45, fontsize=8)

plt.suptitle('MAE SWE per bacino — fold LOYO out-of-sample', fontsize=13)
plt.tight_layout()
plt.savefig(FIG_DIR / 'mae_per_basin.png', dpi=150, bbox_inches='tight')
plt.show()
print('Salvato: mae_per_basin.png')

In [ ]:
# ── Plot: SWE predetto vs osservato per bacino (anni test) ────────────────────
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for ax, (name, info) in zip(axes, BASINS.items()):
    ts = basin_ts[name]
    ax.plot(ts['year'], ts['obs_mean'],  'o-', color='k', label='SPORTLIS', lw=2)
    ax.plot(ts['year'], ts['pred_mean'], 's--', color=info['color'], label='U-Net', lw=2)
    ax.set_title(info['label'], fontsize=10)
    ax.set_xlabel('Anno test'); ax.set_ylabel('SWE medio (mm)')
    ax.legend(fontsize=8)
    ax.set_xticks(ts['year']); ax.set_xticklabels(ts['year'], rotation=45, fontsize=8)

plt.suptitle('SWE medio per bacino: U-Net vs SPORTLIS (fold LOYO)', fontsize=13)
plt.tight_layout()
plt.savefig(FIG_DIR / 'swe_pred_vs_obs_basin.png', dpi=150, bbox_inches='tight')
plt.show()
print('Salvato: swe_pred_vs_obs_basin.png')

In [ ]:
# ── Proiezione anni futuri (2022+) ───────────────────────────────────────────
# Auto-rileva anni disponibili oltre il 2024 guardando ZARR_DIR e MEMMAP_DIR.
# Per aggiungere 2025/2026:
#   1. Nel training notebook, esegui download NARR + build_year_zarr + build_memmap_year
#      per ciascun anno. NARR su PSL è disponibile fino al 2021; per anni successivi
#      La cella 8 del training notebook gestisce anche 2025/2026 (NARR PSL aggiornato).
#   2. Torna qui e riesegui questa cella.

import csv
PROJ_FOLD  = 2020
PROJ_CSV   = OUTPUT_DIR / 'proj_basin_results.csv'
SWE_MAX_MM = 3000.0
TIME_CHUNK = 32
BATCH_SIZE = 32

# ── Auto-rileva anni con zarr O memmap oltre 2024 ────────────────────────────
import re
def _years_in_dir(d, pattern):
    if not d.exists(): return set()
    return {int(m.group(1)) for f in d.iterdir()
            if (m := re.match(pattern, f.name))}

years_zarr   = _years_in_dir(ZARR_DIR,   r'sportlis_narr_pp_(\d{4})\.zarr')
years_memmap = _years_in_dir(MEMMAP_DIR, r'y(\d{4})_feat\.npy')
years_future = sorted((years_zarr | years_memmap) - set(YEARS_AVAILABLE))

print(f'Anni con zarr:   {sorted(years_zarr   - set(YEARS_AVAILABLE))}')
print(f'Anni con memmap: {sorted(years_memmap - set(YEARS_AVAILABLE))}')
print(f'Anni proiettabili: {years_future}')

if not years_future:
    print('\n⚠  Nessun anno futura disponibile.')
    print('   Per aggiungere dati 2025/2026, nel training notebook esegui:')
    print('   for year in [2025, 2026]:')
    print('       download_narr_year(year)   # richiede NARR o ERA5 disponibile')
    print('       build_year_zarr(year)')
    print('       build_memmap_year(year)')
    print('\n   NARR PSL è aggiornato fino ad aprile 2026:')
    print('   https://downloads.psl.noaa.gov/NARR/monolevel/')
    print('   → air.2m.2025.nc (253MB completo), air.2m.2026.nc (65MB fino ad apr 2026)')
    print('\n   Nel training notebook (sportlis_unet_extended_1985_2024.ipynb),')
    print('   esegui la cella 8 "DOWNLOAD NARR + BUILD ZARR/MEMMAP ANNI DI PROIEZIONE".')
    print('   Poi torna qui e riesegui questa cella.')
else:
    # Resume da CSV
    if PROJ_CSV.exists():
        proj_done = set(int(x) for x in pd.read_csv(PROJ_CSV)['year'].unique())
    else:
        proj_done = set()

    ckpt_path = OUTPUT_DIR / CHECKPOINT_TEMPLATE.format(test=PROJ_FOLD)
    if not ckpt_path.exists():
        print(f'Checkpoint fold {PROJ_FOLD} non trovato: {ckpt_path}')
    else:
        mean_d, std_d = load_stats(PROJ_FOLD)
        lat2d, lon2d  = np.meshgrid(lat_1d, lon_1d, indexing='ij')
        lat_n = ((lat2d-lat2d.mean())/(lat2d.std() or 1.0)).astype(np.float32)
        lon_n = ((lon2d-lon2d.mean())/(lon2d.std() or 1.0)).astype(np.float32)

        for py in years_future:
            if py in proj_done:
                print(f'{py}: già calcolato, skip.'); continue
            mm_path  = MEMMAP_DIR / f'y{py}_feat.npy'
            msk_path = MEMMAP_DIR / f'y{py}_mask.npy'
            if not mm_path.exists():
                print(f'{py}: zarr presente ma memmap mancante.')
                print(f'   Esegui nel training notebook: build_memmap_year({py})')
                continue

            print(f'\nProiezione {py} (modello fold={PROJ_FOLD})...')
            mm  = np.lib.format.open_memmap(str(mm_path),  mode='r')
            msk = np.lib.format.open_memmap(str(msk_path), mode='r')
            T = mm.shape[0]; n_feat = len(INPUT_VARS)

            model = UNet(N_IN_CHANNELS, dropout=DROPOUT_P).to(device)
            model.load_state_dict(torch.load(str(ckpt_path), map_location=device))
            model.eval()
            bacc = {name: {'sp':0.0,'n':0} for name in BASINS}

            for tc0 in range(0, T, TIME_CHUNK):
                tc1 = min(tc0+TIME_CHUNK, T); tc = tc1-tc0
                X = np.empty((tc, N_IN_CHANNELS, EXT_H, EXT_W), dtype=np.float32)
                for i, v in enumerate(INPUT_VARS):
                    X[:,i] = (mm[tc0:tc1,i].astype(np.float32)-mean_d[v])/max(std_d[v],1e-6)
                X[:,n_feat]=lat_n[None]; X[:,n_feat+1]=lon_n[None]
                X[:,n_feat+2:]=topo_tensor[None]
                pb = np.empty((tc, EXT_H, EXT_W), dtype=np.float32)
                with torch.no_grad():
                    for b0 in range(0,tc,BATCH_SIZE):
                        b1=min(b0+BATCH_SIZE,tc)
                        pb[b0:b1]=model(torch.from_numpy(X[b0:b1]).to(device)).cpu().numpy()[:,0]
                del X
                pb_mm=np.expm1(np.clip(pb,0,np.log1p(SWE_MAX_MM))); del pb
                msk_c=(msk[tc0:tc1] if msk.ndim==3 else msk[tc0:tc1,0]).astype(bool)
                for name in BASINS:
                    valid=msk_c & basin_masks[name][None]
                    if not valid.any(): continue
                    b=bacc[name]
                    b['sp']+=float(pb_mm[valid].sum()); b['n']+=int(valid.sum())
                del pb_mm, msk_c; gc.collect()
                if torch.cuda.is_available(): torch.cuda.empty_cache()

            del model, mm, msk; gc.collect()
            rows=[]
            for name in BASINS:
                b=bacc[name]; n=max(b['n'],1)
                rows.append(dict(year=py,basin=name,pred_mean=b['sp']/n,n_pixels=b['n']))
                print(f"  {BASINS[name]['label']:30s}: pred={b['sp']/n:6.1f} mm  n={b['n']:,}")
            wh=not PROJ_CSV.exists()
            with open(PROJ_CSV,'a',newline='') as f:
                w=csv.DictWriter(f,fieldnames=rows[0].keys())
                if wh: w.writeheader()
                w.writerows(rows)
            proj_done.add(py); print(f'  Salvato → {PROJ_CSV.name}')

# ── Ricostruisce proj_basin_ts per il plot ────────────────────────────────────
proj_basin_ts = {name:{'year':[],'pred_mean':[]} for name in BASINS}
if PROJ_CSV.exists():
    for _, row in pd.read_csv(PROJ_CSV).sort_values('year').iterrows():
        n=row['basin']
        if n in proj_basin_ts:
            proj_basin_ts[n]['year'].append(int(row['year']))
            proj_basin_ts[n]['pred_mean'].append(float(row['pred_mean']))
print(f'\nAnni proiezione disponibili: {sorted(proj_done) if PROJ_CSV.exists() else []}')


In [ ]:
# ── Plot combinato: LOYO test years + proiezione 2025-2026 ────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for ax, (name, info) in zip(axes, BASINS.items()):
    ts  = basin_ts[name]
    pts = proj_basin_ts[name]

    ax.plot(ts['year'], ts['obs_mean'],  'o-', color='k',
            label='SPORTLIS', lw=2, ms=6, zorder=3)
    ax.plot(ts['year'], ts['pred_mean'], 's--', color=info['color'],
            label='U-Net LOYO', lw=2, ms=6, zorder=3)

    if pts['year']:
        link_x = ([ts['year'][-1]] if ts['year'] else []) + pts['year']
        link_y = ([ts['pred_mean'][-1]] if ts['pred_mean'] else []) + pts['pred_mean']
        ax.plot(link_x, link_y, '^-', color=info['color'],
                lw=2, ms=9, alpha=0.85, label='U-Net proiezione', zorder=4)
        ax.axvspan(2024.5, max(pts['year'])+0.5,
                   alpha=0.07, color=info['color'])

    ax.axvline(2024.5, color='gray', lw=1, ls=':', alpha=0.6)
    ax.set_title(info['label'], fontsize=10, fontweight='bold')
    ax.set_xlabel('Anno'); ax.set_ylabel('SWE medio (mm)')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

plt.suptitle(
    'SWE medio per bacino: SPORTLIS vs U-Net (LOYO out-of-sample) + Proiezione 2025-2026',
    fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / 'basin_ts_projection.png', dpi=150, bbox_inches='tight')
plt.show()
print('Salvato: basin_ts_projection.png')
